# 04 — Compositions: CLIP-guided Diffusion · Transformer+CLIP+Diffusion · GAN vs Diffusion

This notebook demonstrates multi-model composition pipelines.

## Composition overview

```
  CLIP-guided Diffusion
  text prompt → CLIP text encoder → guidance signal
                                        ↓
  x_T (noise) ──[UNet + CLIP grad]──▶ x_0 (image)

  Transformer + CLIP + Diffusion
  LM generates k prompt candidates
      ↓ CLIP ranks vs desired class embedding
  best prompt → CFG diffusion → image

  GAN vs Diffusion comparison
  Same MNIST data, same config → train both → compare samples + diversity
  Diversity metric: mean per-image pixel variance
```

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
if IN_COLAB:
    !pip install -e . -q

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))
from viz import plot_metrics, show_image_grid
from mini_networks.colab.launcher import run_composition

---
## 1. CLIP-guided Diffusion

Uses CLIP cosine similarity as a gradient signal to steer the diffusion
denoising process toward images matching a text prompt.

In [ ]:
cgd_logger = run_composition('clip_guided_diffusion', fast_demo=True)

In [ ]:
plot_metrics(cgd_logger, keys=['clip_loss', 'diffusion_loss'])

In [ ]:
from mini_networks.compositions.clip_guided_diffusion import CLIPGuidedDiffusionConfig, CLIPGuidedDiffusion
from mini_networks.core.logging.logger import Logger
import tempfile

config = CLIPGuidedDiffusionConfig(fast_demo=True)
comp   = CLIPGuidedDiffusion()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'cgd_demo')
    comp.train(config, logger)

samples = comp.sample(config, n_samples=8)
show_image_grid(samples, title='CLIP-guided diffusion samples', nrow=4)

---
## 2. Transformer + CLIP + Diffusion

Three-stage pipeline:
1. Train TransformerLM on Shakespeare
2. Train CLIP on MNIST image-text pairs
3. Train conditioned UNet  

At inference: LM generates prompt candidates → CLIP ranks them → Diffusion generates image.

In [ ]:
tcd_logger = run_composition('transformer_clip_diffusion', fast_demo=True)

In [ ]:
plot_metrics(tcd_logger, keys=['lm_loss', 'clip_loss', 'diffusion_loss'])

In [ ]:
from mini_networks.compositions.transformer_clip_diffusion import (
    TransformerCLIPDiffusionConfig, TransformerCLIPDiffusion
)
from mini_networks.core.logging.logger import Logger
import tempfile

config = TransformerCLIPDiffusionConfig(fast_demo=True)
comp   = TransformerCLIPDiffusion()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'tcd_demo')
    comp.train(config, logger)

# Generate image from a text seed
image = comp.generate_image('digit zero')
show_image_grid(image.unsqueeze(0), title='TransformerCLIPDiffusion generated image')

# Rank prompts by class
class_id, scores = comp.rank_prompts_by_class(['digit zero', 'digit nine', 'the number one'])
print(f'Best class: {class_id}  scores: {[f"{s:.3f}" for s in scores]}')

---
## 3. GAN vs Diffusion comparison

Side-by-side educational comparison:
- GAN: fast inference, unstable training
- DDPM: stable MSE loss, slower inference (T steps)

**Diversity metric:** mean per-image pixel variance (higher = more varied samples)

In [ ]:
from mini_networks.compositions.gan_diffusion_comparison import GANDiffusionConfig, GANDiffusionComparison
from mini_networks.core.logging.logger import Logger
import tempfile

config = GANDiffusionConfig(fast_demo=True)
comp   = GANDiffusionComparison()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'gd_demo')
    comp.train_gan(config, logger)
    comp.train_diffusion(config, logger)
    plot_metrics(logger, keys=['g_loss', 'd_loss', 'diffusion_loss'])

In [ ]:
result = comp.compare(config, n_samples=8)

print(f"{'Metric':<25} {'GAN':>10} {'Diffusion':>12}")
print('-' * 50)
print(f"{'Diversity (pixel var)':<25} {result['gan_diversity']:>10.4f} {result['diffusion_diversity']:>12.4f}")

In [ ]:
import torch

gan_samples  = result['gan_samples']
diff_samples = result['diffusion_samples']

# Side-by-side: GAN on left, Diffusion on right
combined = torch.cat([gan_samples[:4], diff_samples[:4]], dim=0)
show_image_grid(combined, title='GAN (top row) vs Diffusion (bottom row)', nrow=4)

---
## 4. Checkpoint roundtrip — all composition models

Shows that `load_checkpoint()` works for all model types.

In [ ]:
from mini_networks.models.diffusion.config import DiffusionConfig
from mini_networks.models.diffusion.trainer import DDPMTrainer
from mini_networks.models.gan.config import GANConfig
from mini_networks.models.gan.trainer import GANTrainer

# Diffusion
d_config = DiffusionConfig(fast_demo=True)
d_trainer = DDPMTrainer()
d_trainer.load_checkpoint(d_config, diff_artifacts)  # from section 2
d_result = d_trainer.infer(d_config, {'n_samples': 4})
print(f'Diffusion checkpoint: samples shape = {d_result["samples"].shape}')

print('\n✓ All checkpoint roundtrips successful!')